# Bili Studio Transfer-only — Kaggle

เว็บสำหรับ **ดาวน์โหลดไฟล์ → อัปโหลดไฟล์ต้นฉบับไป Google Drive** เท่านั้น ไม่มีการแปลง HLS และไม่ต้องเปิด GPU

1. เปิด Internet ใน Kaggle Settings และแนบ private Dataset ที่มี `rclone.conf`
2. รันเซลล์ตามลำดับ แก้ `RCLONE_TARGET` ถ้า remote/folder ของคุณต่างจากค่าเริ่มต้น
3. เปิดลิงก์เว็บจากเซลล์ tunnel แล้วกรอก access token จากเซลล์ server
4. ดาวน์โหลดจาก Google Drive หรือ BitTorrent; Bilibili ใช้ได้ก็ต่อเมื่อมี Bili23 MCP แยกต่างหาก
5. เลือกไฟล์ในแท็บ Files แล้วกด Upload (ไม่มี auto-upload เพื่อป้องกันการคัดลอกผิดไฟล์)

ข้อมูลอยู่ใน `/kaggle/tmp/bili-downloads` และจะหายเมื่อ Kaggle session สิ้นสุด โปรดอัปโหลดให้เสร็จก่อนปิด session. Notebook นี้ clone โค้ดจาก GitHub; ต้อง push เวอร์ชันที่มี Transfer-only mode ก่อนจึงจะรันเซลล์ setup ผ่าน


In [ ]:
import os, shutil, subprocess
from pathlib import Path

RCLONE_TARGET = 'metube:tube'  # เปลี่ยนเป็น remote:folder ของคุณ
WEB_PORT = 8788
BILI_ROOT = Path('/kaggle/working/bili-pipeline-transfer-only-v1')
BILI_DOWNLOADS = Path('/kaggle/tmp/bili-downloads')
BILI_JOBS = Path('/kaggle/working/bili-transfer-jobs')
BILI_DOWNLOADS.mkdir(parents=True, exist_ok=True)
BILI_JOBS.mkdir(parents=True, exist_ok=True)

configs = sorted(Path('/kaggle/input').rglob('rclone.conf'))
if len(configs) != 1:
    raise RuntimeError(f'Attach exactly one private rclone.conf Dataset; found {len(configs)}: {configs}')
RCLONE_CONFIG_FILE = configs[0]
os.environ['RCLONE_CONFIG'] = str(RCLONE_CONFIG_FILE)
for package in ('rclone', 'aria2c'):
    if not shutil.which(package):
        subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True)
        subprocess.run(['sudo', 'apt-get', 'install', '-y', 'rclone', 'aria2'], check=True)
        break
if not shutil.which('node'):
    raise RuntimeError('Node.js is required for the Drive download worker')
check = subprocess.run(['rclone', 'lsd', RCLONE_TARGET], capture_output=True, text=True, timeout=60)
if check.returncode:
    raise RuntimeError('rclone remote unavailable: ' + check.stderr[-600:])

if not BILI_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NopXx/bili-pipeline.git', str(BILI_ROOT)], check=True)
else:
    if not (BILI_ROOT / '.git').is_dir():
        raise RuntimeError(f'{BILI_ROOT} exists but is not a Git checkout')
    if subprocess.check_output(['git', '-C', str(BILI_ROOT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Transfer-only checkout has local changes; resolve before updating')
    subprocess.run(['git', '-C', str(BILI_ROOT), 'pull', '--ff-only'], check=True)
if 'BILI_TRANSFER_ONLY' not in (BILI_ROOT / 'scripts' / 'bili_web.py').read_text() or not (BILI_ROOT / 'frontend' / 'transfer.html').is_file():
    raise RuntimeError('GitHub repo does not yet contain Transfer-only mode. Push the updated bili-pipeline code first.')
print('Ready:', BILI_ROOT)
print('rclone target:', RCLONE_TARGET, '| downloads:', BILI_DOWNLOADS)
print('No GPU or HLS dependencies needed')


In [ ]:
import json, secrets, time, urllib.error, urllib.request

previous = globals().get('BILI_TRANSFER_PROCESS')
if previous and previous.poll() is None:
    old_token = globals().get('access_token')
    if not old_token:
        raise RuntimeError('Cannot safely restart: previous access token is missing')
    request = urllib.request.Request(f'http://127.0.0.1:{WEB_PORT}/api/jobs', data=b'{}', headers={'X-Token': old_token, 'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=5) as response:
        old_jobs = json.load(response)
    if any(job.get('status') in ('queued', 'running', 'paused') for job in old_jobs):
        raise RuntimeError('Active transfer jobs remain; finish or cancel them before restarting the web server')
    previous.terminate()
    previous.wait(timeout=10)

access_token = secrets.token_urlsafe(32)
server_env = os.environ.copy()
server_env.update({'BILI_WEB_TOKEN': access_token, 'BILI_WEB_HOST': '127.0.0.1', 'BILI_WEB_PORT': str(WEB_PORT), 'BILI_TRANSFER_ONLY': '1', 'BILI_DOWNLOADS_DIR': str(BILI_DOWNLOADS), 'BILI_JOBS_DIR': str(BILI_JOBS), 'BILI_RCLONE_REMOTE': RCLONE_TARGET, 'RCLONE_CONFIG': str(RCLONE_CONFIG_FILE)})
server_log_path = Path('/kaggle/working/bili-transfer-web.log')
with server_log_path.open('w', encoding='utf-8') as log:
    BILI_TRANSFER_PROCESS = subprocess.Popen(['python3', str(BILI_ROOT / 'scripts' / 'bili_web.py')], cwd=BILI_ROOT, env=server_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(30):
    if BILI_TRANSFER_PROCESS.poll() is not None:
        raise RuntimeError('Bili Studio exited: ' + server_log_path.read_text(errors='replace')[-1200:])
    try:
        request = urllib.request.Request(f'http://127.0.0.1:{WEB_PORT}/api/health', data=b'{}', headers={'X-Token': access_token, 'Content-Type': 'application/json'})
        with urllib.request.urlopen(request, timeout=2) as response:
            health = json.load(response)
        if not health.get('transfer_only'):
            raise RuntimeError('Server did not enter Transfer-only mode')
        print('Transfer-only web is ready. Free space: %.1f GiB' % (health['disk_free'] / 1024**3))
        print('ACCESS_TOKEN:', access_token)
        print('Keep this token private; clear notebook outputs before sharing.')
        break
    except urllib.error.URLError:
        time.sleep(1)
else:
    raise RuntimeError('Server did not become healthy: ' + server_log_path.read_text(errors='replace')[-1200:])


In [ ]:
import platform, re, socket

if BILI_TRANSFER_PROCESS.poll() is not None:
    raise RuntimeError('Start the Transfer-only server first')
cloudflared = Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    if platform.machine().lower() not in ('x86_64', 'amd64'):
        raise RuntimeError('This cloudflared binary supports x86_64 only')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
subprocess.run([str(cloudflared), '--version'], check=True, capture_output=True)
tunnel_log_path = Path('/kaggle/working/bili-transfer-tunnel.log')
tunnel = globals().get('BILI_TRANSFER_TUNNEL')
if not tunnel or tunnel.poll() is not None:
    with tunnel_log_path.open('w', encoding='utf-8') as log:
        BILI_TRANSFER_TUNNEL = subprocess.Popen([str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{WEB_PORT}'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
tunnel_url = None
for attempt in range(120):
    log_text = tunnel_log_path.read_text(encoding='utf-8', errors='replace')
    if BILI_TRANSFER_TUNNEL.poll() is not None:
        raise RuntimeError('Tunnel exited: ' + log_text[-1200:])
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match and 'Registered tunnel connection' in log_text:
        try:
            socket.getaddrinfo(match.group(0).split('//', 1)[1], 443, type=socket.SOCK_STREAM)
            tunnel_url = match.group(0)
            break
        except socket.gaierror:
            pass
    if attempt in (20, 60): print('Waiting for tunnel and DNS…')
    time.sleep(1)
if not tunnel_url:
    raise RuntimeError('Tunnel not ready; see ' + str(tunnel_log_path))
print('Open Transfer-only web:', tunnel_url)
print('Enter ACCESS_TOKEN from the previous cell. Keep this Kaggle session running.')


In [ ]:
# Optional: stop only the processes started by this notebook. Finish active jobs first.
for name in ('BILI_TRANSFER_TUNNEL', 'BILI_TRANSFER_PROCESS'):
    process = globals().get(name)
    if process and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)
        print(name, 'stopped')
